<a href="https://colab.research.google.com/github/antoniolopez02-oss/AAI2025/blob/2026Fall/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/antoniolopez02-oss/AAI2025/blob/dev/ML/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

            # Part 1: House Price Prediction


This notebook trains a linear regression model to predict a house price from
square footage and broad location. It uses 1,460 real home sales from the
[OpenML House Prices dataset](https://www.openml.org/search?type=data&status=active&id=42165).

Run the cells from top to bottom in Google Colab.

## 1. Load and check the data

The notebook first looks for the CSV locally. When opened directly
from GitHub in Colab, it reads the same CSV from the repository.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Data source: OpenML House Prices dataset (Ames, Iowa), data ID 42165.
# https://www.openml.org/search?type=data&status=active&id=42165
# GrLivArea was renamed square_footage and SalePrice was renamed price.
# Ames neighborhoods were grouped into Downtown, Rural, and Suburb.
DATA_FILENAME = "ames_housing_prices.csv"
DATA_URL = (
    "https://raw.githubusercontent.com/antoniolopez02-oss/"
    "AAI2025/dev/ML/ames_housing_prices.csv"
)

local_paths = [Path(DATA_FILENAME), Path("ML") / DATA_FILENAME]
data_source = next((path for path in local_paths if path.exists()), DATA_URL)
data = pd.read_csv(data_source)

required_columns = ["square_footage", "location", "price"]
missing_columns = set(required_columns) - set(data.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

data = data[required_columns].dropna().copy()
data = data[(data["square_footage"] > 0) & (data["price"] > 0)]
if len(data) < 100:
    raise ValueError("The assignment requires at least 100 valid records.")

print(f"Valid housing records loaded: {len(data):,}")
data.head()

HTTPError: HTTP Error 404: Not Found

## 2. Prepare and train the model

`OneHotEncoder` changes the location names into numerical columns.
Downtown is used as the comparison category. The remaining data is
split into training and testing groups.

In [ ]:
features = data[["square_footage", "location"]]
target = data["price"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            ["location"],
        )
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)
model.fit(X_train, y_train)
print("Model training complete.")

## 3. Evaluate the model and make the required prediction

In [ ]:
test_predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, test_predictions)
r_squared = r2_score(y_test, test_predictions)

new_house = pd.DataFrame(
    {"square_footage": [2000], "location": ["Downtown"]}
)
predicted_price = model.predict(new_house)[0]

print(
    "Predicted price for a 2,000 sq ft house in Downtown: "
    f"${predicted_price:,.2f}"
)
print(f"Test mean absolute error: ${mae:,.2f}")
print(f"Test R-squared: {r_squared:.3f}")

## 4. Print and explain the coefficients

In [ ]:
encoder = model.named_steps["preprocessor"].named_transformers_["location"]
feature_names = list(encoder.get_feature_names_out(["location"]))
feature_names.append("square_footage")
coefficients = model.named_steps["regressor"].coef_
coefficient_by_feature = dict(zip(feature_names, coefficients))

print("Model coefficients:")
for feature, coefficient in coefficient_by_feature.items():
    print(f"  {feature}: {coefficient:,.2f}")

square_footage_effect = coefficient_by_feature["square_footage"]
print("\nPlain-language explanation:")
print(
    "  Holding location constant, each additional square foot is "
    f"associated with about ${square_footage_effect:,.2f} in predicted price."
)

for location in ("Rural", "Suburb"):
    location_effect = coefficient_by_feature[f"location_{location}"]
    direction = "higher" if location_effect >= 0 else "lower"
    print(
        f"  A similar-size house in {location} is predicted to be "
        f"${abs(location_effect):,.2f} {direction} than one in Downtown."
    )